# Loss Normalization Experiment: Evaluation

Evaluate the fitted models from the `fusionreg × l2reg` grid against
ground-truth mutation effects.

**Key outputs:**
1. Sparsity vs fusionreg (one curve per l2reg)
2. β correlation with ground truth vs fusionreg (one curve per l2reg)
3. `fit_summary.csv` — per-model summary table

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import os
import pickle
import sys

sys.path.insert(0, "notebooks")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import multidms
from multidms.model_collection import ModelCollection

from _common import load_config

In [ ]:
config_path = "config/config.yaml"
output_dir = None

In [ ]:
config = load_config(config_path)
exp = config["experiment"]
if output_dir is None:
    output_dir = exp["output_dir"]

true_effects_path = exp["data"]["true_effects"]
print(f"Output directory: {output_dir}")

## Load data

In [ ]:
fit_collection_df = pickle.load(
    open(os.path.join(output_dir, "fit_collection.pkl"), "rb")
)
true_effects = pd.read_csv(true_effects_path)
print(f"Loaded {len(fit_collection_df)} fitted models")
print(f"Ground truth: {len(true_effects)} mutations")

## Build per-model summary

For each fitted model, compute:
- Shift sparsity (fraction of zero shifts)
- Pearson correlation of β with ground truth
- Pearson correlation of shifts with ground truth

In [ ]:
def compute_model_summary(row, true_effects):
    """Compute accuracy metrics for a single fitted model."""
    model = row["model"]
    mc = ModelCollection(pd.DataFrame([row]))
    muts_df = mc.get_mutations_df().reset_index()

    # Merge with ground truth
    merged = muts_df.merge(true_effects, on="mutation", how="inner")

    result = {
        "fusionreg": row["fusionreg"],
        "l2reg": row["l2reg"],
        "dataset_name": row["dataset_name"],
        "library": row.get("library", ""),
        "measurement_type": row.get("measurement_type", ""),
        "fit_time": row.get("fit_time", np.nan),
    }

    # β correlation (reference condition)
    if "beta" in merged.columns and "beta_h1" in merged.columns:
        mask = merged["beta"].notna() & merged["beta_h1"].notna()
        if mask.sum() > 2:
            result["beta_corr"] = merged.loc[mask, "beta"].corr(
                merged.loc[mask, "beta_h1"]
            )

    # Shift correlation and sparsity (non-reference condition)
    shift_col = [c for c in merged.columns if "shift" in c.lower() and "predicted" not in c.lower()]
    true_shift_col = "beta_h2" if "beta_h2" in merged.columns else None
    if shift_col and true_shift_col:
        sc = shift_col[0]
        mask = merged[sc].notna() & merged[true_shift_col].notna()
        if mask.sum() > 2:
            result["shift_corr"] = merged.loc[mask, sc].corr(
                merged.loc[mask, true_shift_col]
            )
        # Sparsity: fraction of exactly zero shifts
        result["shift_sparsity"] = (merged[sc] == 0).mean()

    # True sparsity for reference
    if "shifted" in merged.columns:
        result["true_sparsity"] = 1.0 - merged["shifted"].mean()
    elif true_shift_col:
        # If no 'shifted' column, approximate: mutations with beta_h1 == beta_h2
        result["true_sparsity"] = (
            (merged["beta_h1"] - merged[true_shift_col]).abs() < 1e-10
        ).mean()

    return result


summaries = []
for _, row in fit_collection_df.iterrows():
    summaries.append(compute_model_summary(row, true_effects))

summary_df = pd.DataFrame(summaries)
summary_df.to_csv(os.path.join(output_dir, "fit_summary.csv"), index=False)
print(f"Summary: {len(summary_df)} rows")
summary_df.head(10)

## 1. Sparsity vs fusionreg

One curve per `l2reg` value. Shows fraction of zero-shift parameters as a
function of fusion regularization strength, compared against true sparsity.

In [ ]:
# Average across datasets (libraries × measurement types) for each (fusionreg, l2reg)
agg = summary_df.groupby(["fusionreg", "l2reg"]).agg(
    shift_sparsity=("shift_sparsity", "mean"),
    true_sparsity=("true_sparsity", "mean"),
).reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
for l2val, grp in agg.groupby("l2reg"):
    grp_sorted = grp.sort_values("fusionreg")
    ax.plot(
        grp_sorted["fusionreg"],
        grp_sorted["shift_sparsity"],
        "o-",
        label=f"l2reg={l2val:.1e}",
    )

# True sparsity reference line
if "true_sparsity" in agg.columns and agg["true_sparsity"].notna().any():
    true_sp = agg["true_sparsity"].mean()
    ax.axhline(true_sp, color="k", linestyle="--", alpha=0.5, label=f"true sparsity={true_sp:.2f}")

ax.set_xlabel("fusionreg")
ax.set_ylabel("Shift sparsity (fraction zero)")
ax.set_title("Shift sparsity vs fusion regularization")
ax.legend()
ax.set_xscale("symlog", linthresh=1e-6)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "sparsity_vs_fusionreg.pdf"))
plt.show()

## 2. β correlation with ground truth vs fusionreg

Pearson correlation between inferred and true mutation effects, broken out by `l2reg`.

In [ ]:
agg_corr = summary_df.groupby(["fusionreg", "l2reg"]).agg(
    beta_corr=("beta_corr", "mean"),
    shift_corr=("shift_corr", "mean"),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for l2val, grp in agg_corr.groupby("l2reg"):
    grp_sorted = grp.sort_values("fusionreg")
    axes[0].plot(
        grp_sorted["fusionreg"],
        grp_sorted["beta_corr"],
        "o-",
        label=f"l2reg={l2val:.1e}",
    )
    axes[1].plot(
        grp_sorted["fusionreg"],
        grp_sorted["shift_corr"],
        "s-",
        label=f"l2reg={l2val:.1e}",
    )

axes[0].set_xlabel("fusionreg")
axes[0].set_ylabel("Pearson r (β vs truth)")
axes[0].set_title("β accuracy vs fusion regularization")
axes[0].legend()
axes[0].set_xscale("symlog", linthresh=1e-6)

axes[1].set_xlabel("fusionreg")
axes[1].set_ylabel("Pearson r (shift vs truth)")
axes[1].set_title("Shift accuracy vs fusion regularization")
axes[1].legend()
axes[1].set_xscale("symlog", linthresh=1e-6)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "correlation_vs_fusionreg.pdf"))
plt.show()

## 3. Best model per l2reg

Identify the best (fusionreg, l2reg) combination for each l2reg value.

In [ ]:
if "beta_corr" in agg_corr.columns:
    best = agg_corr.loc[agg_corr.groupby("l2reg")["beta_corr"].idxmax()]
    print("Best β correlation per l2reg:")
    print(best[["l2reg", "fusionreg", "beta_corr"]].to_string(index=False))
    print()

    overall_best = agg_corr.loc[agg_corr["beta_corr"].idxmax()]
    print(f"Overall best: fusionreg={overall_best['fusionreg']:.2e}, "
          f"l2reg={overall_best['l2reg']:.2e}, "
          f"beta_corr={overall_best['beta_corr']:.4f}")
else:
    print("No beta_corr data available")